# Detección de Huecos en Carreteras — Pothole Detection

**Dataset:** `taroii/pothole-detection`  
**Modelo:** InceptionV3 con Transfer Learning  
**Tarea:** Clasificación binaria: `pothole` vs `no pothole`

Este notebook está organizado para trabajar con varios archivos del proyecto en la carpeta `src/`.


## 0. Configuración para Colab

En Colab, primero asegúrate de estar dentro de la carpeta raíz del proyecto. Si subiste el ZIP, descomprímelo y entra a la carpeta. Si clonaste desde GitHub, entra al repo clonado.


In [ ]:
# En Colab, ajusta esta ruta si tu carpeta tiene otro nombre.
# Si ya estás en la carpeta correcta, esta celda no cambia nada.
import os, sys

possible_paths = [
    "/content/proyecto_deep_arreglado",
    "/content/proyecto_deep",
    os.getcwd(),
]

for path in possible_paths:
    if os.path.exists(os.path.join(path, "src")):
        os.chdir(path)
        break

project_root = os.getcwd()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print("Directorio actual:", os.getcwd())
print("Archivos en raíz:", os.listdir())
print("Archivos en src:", os.listdir("src") if os.path.exists("src") else "No existe src")


In [ ]:
# Instala dependencias si estás en Colab o si falta alguna librería.
# En Colab puedes descomentar esta línea:
# !pip install -r requirements.txt


## 1. Imports y reproducibilidad

In [ ]:
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

from src.data_loaders import (
    load_pothole_dataset,
    create_dataloaders,
    plot_class_distribution,
    show_samples_by_class,
    image_dimension_report,
    pixel_statistics_report,
)
from src.models import build_pothole_inception, unfreeze_last_layers, count_trainable_parameters
from src.model_training import train_model, evaluate, classification_metrics
from src.utils import plot_training_curves, plot_confusion_matrix, show_predictions
from src.xai import attach_hooks, run_forward, run_backward, compute_gradcam, resize_cam, tensor_to_uint8, overlay_cam

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print(f"PyTorch: {torch.__version__}")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


## 2. Carga del dataset

Se carga el dataset de Hugging Face y se detectan automáticamente la columna de imagen, la columna de etiqueta y los nombres de las clases.


In [ ]:
DATASET_NAME = "taroii/pothole-detection"

info = load_pothole_dataset(DATASET_NAME)
ds = info.dataset

print("Splits disponibles:", list(ds.keys()))
print("Columna imagen:", info.image_key)
print("Columna etiqueta:", info.label_key)
print("Clases:", info.class_names)
print("Número de clases:", info.num_classes)

for split in ds.keys():
    print(f"{split}: {len(ds[split])} imágenes")


## 3. EDA — Análisis exploratorio de datos

Esta sección conserva la lógica del EDA del notebook original: distribución de clases por split, ejemplos visuales, dimensiones de las imágenes y estadísticas de intensidad por canal RGB.


In [ ]:
os.makedirs("output", exist_ok=True)

plot_class_distribution(info, save_path="output/class_distribution.png")


In [ ]:
show_samples_by_class(info, split="train", samples_per_class=4, save_path="output/sample_images.png")


In [ ]:
df_dims = image_dimension_report(info, split="train", n_samples=100, save_path="output/image_dimensions.png")
df_dims.head()


In [ ]:
pixel_stats = pixel_statistics_report(info, split="train", n_samples=200, save_path="output/pixel_distribution.png")
pixel_stats


## 4. Preprocesamiento y DataLoaders

Se redimensionan las imágenes a `299 x 299`, tamaño esperado por InceptionV3, se aplica normalización de ImageNet y se usa data augmentation solo en entrenamiento.


In [ ]:
IMAGE_SIZE = 299
BATCH_SIZE = 32
NUM_WORKERS = 2

train_loader, val_loader, test_loader, train_ds, val_ds, test_ds, info = create_dataloaders(
    dataset_name=DATASET_NAME,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
)

CLASS_NAMES = info.class_names
NUM_CLASSES = info.num_classes

print("Clases:", CLASS_NAMES)
print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))


## 5. Modelo CNN con Transfer Learning

Se usa InceptionV3 preentrenado en ImageNet. Primero se congelan las capas base y se entrena solo el clasificador final.


In [ ]:
model = build_pothole_inception(num_classes=NUM_CLASSES, dropout=0.4, feature_extract=True)
model = model.to(DEVICE)

print(model.__class__.__name__)
print("Parámetros entrenables:", count_trainable_parameters(model))


## 6. Fase 1 — Feature Extraction

In [ ]:
os.makedirs("models", exist_ok=True)
os.makedirs("output", exist_ok=True)

EPOCHS_PHASE1 = 5
LR_PHASE1 = 1e-3

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR_PHASE1,
)

print("=== FASE 1: Feature Extraction ===")
train_losses_p1, val_losses_p1, train_accs_p1, val_accs_p1 = train_model(
    model=model,
    model_name="pothole_inception_phase1",
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=DEVICE,
    epochs=EPOCHS_PHASE1,
)


In [ ]:
plot_training_curves(
    train_losses_p1, val_losses_p1,
    train_accs_p1, val_accs_p1,
    title="Fase 1: Feature Extraction",
    save_path="output/curves_phase1.png",
)

test_loss_p1, test_acc_p1 = evaluate(model, test_loader, criterion, DEVICE)
print(f"Test Loss Fase 1: {test_loss_p1:.4f}")
print(f"Test Accuracy Fase 1: {test_acc_p1:.2f}%")


## 7. Fase 2 — Fine-tuning

Se descongelan algunas capas finales del modelo y se entrena con un learning rate más bajo.


In [ ]:
model = unfreeze_last_layers(model, num_layers=40)
print("Parámetros entrenables después de fine-tuning:", count_trainable_parameters(model))

EPOCHS_PHASE2 = 6
LR_PHASE2 = 1e-4

optimizer_ft = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR_PHASE2,
)

print("=== FASE 2: Fine-tuning ===")
train_losses_p2, val_losses_p2, train_accs_p2, val_accs_p2 = train_model(
    model=model,
    model_name="pothole_inception_finetune",
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer_ft,
    device=DEVICE,
    epochs=EPOCHS_PHASE2,
)


In [ ]:
plot_training_curves(
    train_losses_p2, val_losses_p2,
    train_accs_p2, val_accs_p2,
    title="Fase 2: Fine-tuning",
    save_path="output/curves_phase2.png",
)

test_loss_p2, test_acc_p2 = evaluate(model, test_loader, criterion, DEVICE)
print(f"Test Loss Fase 2: {test_loss_p2:.4f}")
print(f"Test Accuracy Fase 2: {test_acc_p2:.2f}%")
print(f"Mejora vs Fase 1: {test_acc_p2 - test_acc_p1:.2f} puntos porcentuales")


## 8. Evaluación final

Se calculan métricas por clase, matriz de confusión y ejemplos visuales de predicciones.


In [ ]:
metrics = classification_metrics(model, test_loader, DEVICE, class_names=CLASS_NAMES)

print(f"Accuracy: {metrics['accuracy']:.2f}%")
print(f"Precision weighted: {metrics['precision_weighted']:.4f}")
print(f"Recall weighted: {metrics['recall_weighted']:.4f}")
print(f"F1 weighted: {metrics['f1_weighted']:.4f}")
print("
Classification report:")
print(metrics["classification_report"])


In [ ]:
plot_confusion_matrix(
    metrics["confusion_matrix"],
    CLASS_NAMES,
    save_path="output/confusion_matrix.png",
)


In [ ]:
show_predictions(
    model=model,
    dataloader=test_loader,
    device=DEVICE,
    class_names=CLASS_NAMES,
    n=8,
    save_path="output/predictions.png",
)


## 9. Grad-CAM — Explicabilidad

Grad-CAM ayuda a revisar si el modelo está mirando la zona del hueco o si se está guiando por señales espurias como sombras, bordes o textura del pavimento.


In [ ]:
# Grad-CAM con una imagen del test set
# Para InceptionV3, una capa profunda útil es Mixed_7c.

model.eval()
if hasattr(model, "aux_logits"):
    model.aux_logits = False

images, labels = next(iter(test_loader))
image = images[0].unsqueeze(0).to(DEVICE)
image.requires_grad_(True)
true_label = int(labels[0])

capture, remove = attach_hooks(model, "Mixed_7c")
output, pred = run_forward(model, image)
run_backward(output, pred)
cam = compute_gradcam(capture["act"], capture["grad"])
cam = resize_cam(cam, images[0].shape[1], images[0].shape[2])
remove()

# Reconstrucción visual de la imagen normalizada
from src.utils import unnormalize
orig = unnormalize(images[0])
orig_uint8 = (orig * 255).astype(np.uint8)
cam_np = cam.detach().cpu().numpy()
overlay, heatmap = overlay_cam(orig_uint8, cam_np)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(orig_uint8)
axes[0].set_title(f"Original
Real: {CLASS_NAMES[true_label]}")
axes[0].axis("off")

axes[1].imshow(heatmap)
axes[1].set_title("Grad-CAM")
axes[1].axis("off")

axes[2].imshow(overlay)
axes[2].set_title(f"Overlay
Pred: {CLASS_NAMES[pred]}")
axes[2].axis("off")

plt.tight_layout()
plt.savefig("output/gradcam_example.png", bbox_inches="tight", dpi=140)
plt.show()


## 10. Resumen para el proyecto

Este pipeline cumple con los componentes principales del proyecto:

- selección y justificación de una base de datos de imágenes;
- EDA con visualizaciones;
- preprocesamiento y aumento de datos;
- implementación de una CNN con transfer learning;
- entrenamiento en dos fases;
- evaluación con múltiples métricas;
- explicabilidad mediante Grad-CAM;
- resultados visuales para usar en la presentación.
